# NHANES 2021–2023 — Exploratory Data Analysis v3
### Model 2: OW_I vs OW_II Binary Classifier — Feature Engineering & Gender Split Analysis

**What changed from v2:** This version adds the full feature engineering pipeline derived from the variable analysis document, plus a dedicated gender split analysis to determine whether training separate male and female models is statistically justified.

**Dataset:** CDC NHANES 2021–2023 cycle. 11,933 participants across demographics, exam, lab, and questionnaire files.

---

**Notebook structure:**
1. Load, merge & sentinel clean (carried forward from v2)
2. Pre-model filters (MEC-exam, adults, exclude pregnant)
3. NHANES refusal code cleaning
4. Structural missingness zero-imputation
5. Feature engineering — all derived features
6. Encoding — binary, ordinal, one-hot
7. Drop columns — leakage, redundant units, zero-variance
8. Target variable & final dataset shape
9. Gender split feasibility analysis
10. Feature summary & export

---
## 1. Load, Merge & Sentinel Clean

We load the master merged CSV produced in v2. The merge was validated (256 columns, 11,933 unique SEQNs, no _x/_y duplicates). We immediately replace the SAS system missing sentinel (5.397605e-79) with NaN — this float is not a real measurement, it is SAS's internal missing value that survives XPT export.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('merged_data/master_merged_data.csv')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Replace SAS system missing sentinel with NaN
for col in df.select_dtypes(include='float').columns:
    df.loc[df[col].abs() < 1e-70, col] = np.nan

print('SAS sentinel values replaced with NaN.')

---
## 2. Pre-Model Filters

Three mandatory filters must be applied before any encoding or feature engineering:

**MEC-examined only (`RIDSTATR == 2`):** Interview-only participants (RIDSTATR = 1) have no physical exam or lab data. They cannot contribute to Model 2 regardless of questionnaire responses.

**Adults only (`RIDAGEYR >= 20`):** BMI classification works differently for children and adolescents (requires age- and sex-specific percentiles). The OW_I/OW_II boundaries only apply to adults. Age 20 also ensures education (DMDEDUC2) and marital status (DMDMARTZ) are populated.

**Exclude pregnant women (`RIDEXPRG == 1` or `3`):** BMI during pregnancy does not reflect stable adiposity. Including pregnant participants would corrupt the OW_I/OW_II boundary. Code 3 (cannot ascertain) is also excluded as ambiguous.

In [ ]:
print(f'Before filters: {len(df):,} rows')

# Filter 1: MEC-examined only
df = df[df['RIDSTATR'] == 2].copy()
print(f'After MEC-exam filter (RIDSTATR==2): {len(df):,} rows')

# Filter 2: Adults 20+
df = df[df['RIDAGEYR'] >= 20].copy()
print(f'After age filter (>=20): {len(df):,} rows')

# Filter 3: Exclude pregnant and cannot-ascertain women
df = df[~df['RIDEXPRG'].isin([1, 3])].copy()
print(f'After pregnancy exclusion: {len(df):,} rows')

print(f'\nSex breakdown: Male={( df["RIAGENDR"]==1).sum():,}  Female={(df["RIAGENDR"]==2).sum():,}')

---
## 3. NHANES Refusal Code Cleaning

Questionnaire variables encode participant refusals and don't-know responses as numeric codes (7, 9, 77, 99, 777, 999, 7777, 9999). These look like real values but carry no clinical meaning. We replace them with NaN **before** any binary encoding — if we encoded first, a refusal (7) would become a valid category.

**Important:** We do NOT apply this to lab columns (LBXGLU, LBXTC, etc.) — there a value of 99 is a legitimate glucose reading. The cleaning map is applied only to the variables documented in the variable analysis.

In [ ]:
nhanes_cleaning_map = {
    # Demographics
    'DMQMILIZ': [7, 9],
    'DMDBORN4': [77, 99],
    'DMDYRUSR': [77, 99],
    'DMDEDUC2': [7, 9],
    'DMDMARTZ': [77, 99],
    # Blood pressure questionnaire
    'BPQ020':   [7, 9],
    'BPQ030':   [7, 9],
    'BPQ080':   [7, 9],
    'BPQ101D':  [9],
    'BPQ150':   [9],
    # Diabetes
    'DIQ010':   [7, 9],
    'DIQ160':   [7, 9],
    'DIQ050':   [7, 9],
    'DIQ070':   [7, 9],
    # Medical conditions
    'MCQ010':   [7, 9],
    'MCQ160A':  [7, 9],
    'MCQ160B':  [7, 9],
    'MCQ160C':  [7, 9],
    'MCQ160D':  [7, 9],
    'MCQ160E':  [7, 9],
    'MCQ160F':  [7, 9],
    'MCQ160L':  [7, 9],
    'MCQ160M':  [7, 9],
    'MCQ160P':  [7, 9],
    'MCQ220':   [7, 9],
    'MCQ500':   [7, 9],
    'MCQ053':   [7, 9],
    'OSQ230':   [7, 9],
    # Kidney/urology
    'KIQ022':   [7, 9],
    'KIQ005':   [7, 9],
    'KIQ010':   [9],
    'KIQ042':   [9],
    'KIQ044':   [7, 9],
    'KIQ052':   [7, 9],
    'KIQ481':   [7, 9],
    # Physical activity
    'PAD790Q':  [7777, 9999],
    'PAD800':   [7777, 9999],
    'PAD810Q':  [7777, 9999],
    'PAD820':   [7777, 9999],
    'PAD680':   [7777, 9999],
    # Alcohol
    'ALQ111':   [7, 9],
    'ALQ121':   [77, 99],
    'ALQ130':   [777, 999],
    'ALQ142':   [77, 99],
    'ALQ151':   [7, 9],
    'ALQ270':   [77, 99],
    'ALQ280':   [77, 99],
    # Smoking
    'SMQ020':   [7, 9],
    'SMQ040':   [7, 9],
    'SMD650':   [777, 999],
    # Weight history
    'WHD020':   [7777, 9999],
    'WHD050':   [7777, 9999],
    'WHQ070':   [7, 9],
    'WHD010':   [7777, 9999],
    # Balance questionnaire
    'BAQ321A':  [7, 9],
    'BAQ321B':  [7, 9],
    'BAQ321C':  [7, 9],
    'BAQ321D':  [7, 9],
    'BAQ341':   [7, 9],
    'BAQ401':   [7, 9],
    'BAQ530':   [77, 99],
    'BAQ550':   [77, 99],
}

cleaned_count = 0
for col, codes in nhanes_cleaning_map.items():
    if col in df.columns:
        before = df[col].notna().sum()
        df[col] = df[col].replace(codes, np.nan)
        after = df[col].notna().sum()
        if before != after:
            cleaned_count += (before - after)

print(f'Refusal/DK codes replaced with NaN: {cleaned_count:,} values across {len(nhanes_cleaning_map)} columns')

---
## 4. Structural Missingness — Zero Imputation

In NHANES, many variables are only asked conditionally. When a participant answers No to a gateway question, all downstream questions are skipped — those NaNs mean **zero/no**, not **unknown**. Treating them as random missing and imputing with median would be wrong (e.g. imputing a median drinking quantity for someone who never drank).

We must impute these as 0 **before** any downstream imputation strategy touches the data.

In [ ]:
# ALQ: never-drinkers get 0 on all downstream drinking variables
never_drinker_mask = df['ALQ111'] == 2
for col in ['ALQ121', 'ALQ130', 'ALQ142', 'ALQ151', 'ALQ270', 'ALQ280']:
    if col in df.columns:
        df.loc[never_drinker_mask, col] = df.loc[never_drinker_mask, col].fillna(0)

# ALQ: non-past-year-drinkers get 0 on binge variables
no_pastyear_mask = df['ALQ121'] == 0
for col in ['ALQ130', 'ALQ142', 'ALQ270', 'ALQ280']:
    if col in df.columns:
        df.loc[no_pastyear_mask, col] = df.loc[no_pastyear_mask, col].fillna(0)

# ALQ: no-binge participants get 0 on heavy/rapid binge variables
no_binge_mask = df['ALQ142'] == 0
for col in ['ALQ270', 'ALQ280']:
    if col in df.columns:
        df.loc[no_binge_mask, col] = df.loc[no_binge_mask, col].fillna(0)

# BPQ: confirmed non-hypertensive get 0 on BP medication
no_htn_mask = df['BPQ020'] == 2
for col in ['BPQ030', 'BPQ150']:
    if col in df.columns:
        df.loc[no_htn_mask, col] = df.loc[no_htn_mask, col].fillna(0)

# KIQ: no/rare leakage gets 0 on leakage amount and type
no_leak_mask = df['KIQ005'].fillna(0) < 2
for col in ['KIQ010', 'KIQ042', 'KIQ044', 'KIQ052']:
    if col in df.columns:
        df.loc[no_leak_mask, col] = df.loc[no_leak_mask, col].fillna(0)

# DIQ: confirmed non-diabetic gets 0 on insulin/meds variables
no_diab_mask = df['DIQ010'] == 2
for col in ['DIQ050', 'DIQ070']:
    if col in df.columns:
        df.loc[no_diab_mask, col] = df.loc[no_diab_mask, col].fillna(0)

# SMQ: never-smoker gets 0 on cigarettes-per-day
never_smoke_mask = df['SMQ020'] == 2
if 'SMD650' in df.columns:
    df.loc[never_smoke_mask, 'SMD650'] = df.loc[never_smoke_mask, 'SMD650'].fillna(0)

# Former smoker (SMQ040==3) also gets 0 on current cigarettes per day
former_smoke_mask = (df['SMQ020'] == 1) & (df['SMQ040'] == 3)
if 'SMD650' in df.columns:
    df.loc[former_smoke_mask, 'SMD650'] = df.loc[former_smoke_mask, 'SMD650'].fillna(0)

print('Structural zero imputation complete.')

---
## 5. Feature Engineering

We now create all derived features before encoding. Derived features are computed from raw columns before those raw columns are dropped or transformed. The order matters — for example, HOMA-IR must be computed before LBXIN is log-transformed.

### 5a — Body Composition Ratios
Waist-to-height ratio is a stronger metabolic risk predictor than waist circumference alone because it accounts for body size. Waist-to-hip ratio captures fat distribution (central vs peripheral). Both are well-established clinical indices for differentiating OW_I from OW_II.

In [ ]:
# Waist-to-height ratio — clinical threshold: >0.5 = elevated metabolic risk
df['WAIST_HT_RATIO'] = df['BMXWAIST'] / df['BMXHT']

# Waist-to-hip ratio — central vs peripheral fat distribution
df['WAIST_HIP_RATIO'] = df['BMXWAIST'] / df['BMXHIP']

print('Body composition ratios:')
print(df[['WAIST_HT_RATIO', 'WAIST_HIP_RATIO']].describe().round(3))

### 5b — Blood Pressure Averaging

NHANES takes three sequential BP readings. The first is typically highest due to the white-coat effect (anxiety at the start of the exam). The second and third are more physiologically stable. We average all three for each of systolic and diastolic rather than keeping six separate columns — the correlation between readings 1, 2, and 3 is 0.93–0.94, meaning they are almost entirely redundant as separate features.

In [ ]:
# Verify white-coat effect — reading 1 should be highest
print('Mean systolic per reading (white-coat verification):')
print(df[['BPXOSY1','BPXOSY2','BPXOSY3']].mean().round(2))

# Average the three readings
df['SYS_BP'] = df[['BPXOSY1','BPXOSY2','BPXOSY3']].mean(axis=1)
df['DIA_BP'] = df[['BPXODI1','BPXODI2','BPXODI3']].mean(axis=1)

# Drop the raw readings — they are now redundant
bp_raw_cols = ['BPXOSY1','BPXOSY2','BPXOSY3','BPXODI1','BPXODI2','BPXODI3',
               'BPXOPLS1','BPXOPLS2','BPXOPLS3','BPAOARM','BPAOCSZ']
df.drop(columns=[c for c in bp_raw_cols if c in df.columns], inplace=True)

print('\nAveraged BP statistics:')
print(df[['SYS_BP','DIA_BP']].describe().round(2))

### 5c — HOMA-IR (Insulin Resistance Index)

HOMA-IR (Homeostatic Model Assessment of Insulin Resistance) is the standard clinical composite for insulin resistance. It is computed from fasting glucose and fasting insulin and is a stronger OW_I vs OW_II discriminator than either measurement alone — because insulin resistance is the mechanistic driver of the metabolic differences between the two classes.

HOMA-IR must be computed **before** log-transforming insulin, since the formula requires the raw insulin value. Clinical interpretation: values above 2.5 indicate insulin resistance.

We then apply log1p to HOMA-IR because the distribution is heavily right-skewed.

In [ ]:
# HOMA-IR = (fasting glucose mg/dL × fasting insulin µU/mL) / 405
# Requires both LBXGLU and LBXIN — only available for fasting subsample
df['HOMA_IR'] = (df['LBXGLU'] * df['LBXIN']) / 405
df['HOMA_IR_log'] = np.log1p(df['HOMA_IR'])

fasting_n = df['HOMA_IR'].notna().sum()
print(f'HOMA-IR computed for {fasting_n:,} fasting participants')
print(f'Insulin resistance (HOMA-IR > 2.5): {(df["HOMA_IR"] > 2.5).sum():,} participants')
print(df['HOMA_IR'].describe().round(3))

### 5d — Lab Transforms (log1p)

Several lab variables are heavily right-skewed due to a small number of extremely high values. We apply log1p (log(1 + x)) rather than log(x) because some values may be zero after imputation. log1p compresses the long tail without distorting the distribution for most participants.

Variables requiring log1p: fasting insulin (LBXIN), ALT (LBXSATSI), AST (LBXSASSI), GGT (LBXSGTSI), total bilirubin (LBXSTB), CPK (LBXSCK).

In [ ]:
log_transform_cols = {
    'LBXIN':    'LBXIN_log',    # Fasting insulin — extremely right-skewed
    'LBXSATSI': 'ALT_log',      # ALT (liver enzyme)
    'LBXSASSI': 'AST_log',      # AST (liver enzyme)
    'LBXSGTSI': 'GGT_log',      # GGT (liver/metabolic syndrome marker)
    'LBXSTB':   'TBILI_log',    # Total bilirubin
    'LBXSCK':   'CPK_log',      # CPK — extreme outliers (max 16,837)
}

for raw_col, new_col in log_transform_cols.items():
    if raw_col in df.columns:
        # Winsorize CPK at 99th pct before log to reduce extreme outlier influence
        if raw_col == 'LBXSCK':
            p99 = df[raw_col].quantile(0.99)
            df[raw_col] = df[raw_col].clip(upper=p99)
        df[new_col] = np.log1p(df[raw_col])
        print(f'{raw_col:12} → {new_col:12} | skew before: {df[raw_col].skew():.2f} → after: {df[new_col].skew():.2f}')

# Drop raw versions of the transformed columns
df.drop(columns=[c for c in log_transform_cols.keys() if c in df.columns], inplace=True)
print('\nRaw versions dropped. Log-transformed columns created.')

### 5e — Biochemistry Ratios

Three clinically meaningful ratios from the BIOPRO_L panel. These capture relationships between markers that are more informative than the individual values alone:

- **AST/ALT ratio:** >2 suggests alcoholic liver disease; <1 suggests non-alcoholic fatty liver (NAFLD), which is common in OW_II
- **BUN/Creatinine ratio:** >20 suggests prerenal causes (dehydration, high protein intake) — both common in obese individuals
- **eGFR:** Estimated glomerular filtration rate using the CKD-EPI formula — more clinically interpretable than raw creatinine for kidney function

In [ ]:
# AST/ALT ratio — must be computed before log transforms of these variables
# We use AST_log and ALT_log: ratio of logs = log(AST) - log(ALT) (valid log property)
if 'AST_log' in df.columns and 'ALT_log' in df.columns:
    df['AST_ALT_ratio_log'] = df['AST_log'] - df['ALT_log']

# BUN/Creatinine ratio
if 'LBXSBU' in df.columns and 'LBXSCR' in df.columns:
    df['BUN_CREATININE_RATIO'] = df['LBXSBU'] / df['LBXSCR'].replace(0, np.nan)

# eGFR using CKD-EPI equation (2021 race-free version)
# eGFR = 142 × min(Scr/κ, 1)^α × max(Scr/κ, 1)^(-1.200) × 0.9938^Age
# κ = 0.7 (female) or 0.9 (male); α = -0.241 (female) or -0.302 (male)
if 'LBXSCR' in df.columns:
    is_female = df['RIAGENDR'] == 2
    kappa = np.where(is_female, 0.7, 0.9)
    alpha = np.where(is_female, -0.241, -0.302)
    scr_ratio = df['LBXSCR'] / kappa
    df['eGFR'] = (
        142
        * np.minimum(scr_ratio, 1) ** alpha
        * np.maximum(scr_ratio, 1) ** (-1.200)
        * (0.9938 ** df['RIDAGEYR'])
    )
    print(f'eGFR computed. Mean: {df["eGFR"].mean():.1f}  |  Median: {df["eGFR"].median():.1f}')

print('Biochemistry ratios computed.')

### 5f — Physical Activity (Standardise to Minutes per Week)

PAD790Q and PAD810Q record activity frequency in mixed units — participants reported in days-per-day, days-per-week, or days-per-month depending on how they naturally think about their activity. The unit flag columns (PAD790U, PAD810U) tell us which unit each participant used. We standardise everything to minutes-per-week.

In [ ]:
def standardise_to_weekly(freq_col, unit_col, minutes_col, df):
    """
    Convert physical activity frequency to weekly minutes.
    Unit codes: D=per day, W=per week, M=per month, Y=per year
    Returns series of weekly minutes.
    """
    freq = df[freq_col].copy()
    unit = df[unit_col].astype(str).str.strip("b'").str.strip()
    mins = df[minutes_col].copy()

    # Convert frequency to per-week
    freq_weekly = freq.copy()
    freq_weekly = freq_weekly.where(unit != 'D', freq * 7)
    freq_weekly = freq_weekly.where(unit != 'M', freq / 4.33)
    freq_weekly = freq_weekly.where(unit != 'Y', freq / 52)
    # unit 'W' stays as-is

    return (freq_weekly * mins).clip(lower=0)

if all(c in df.columns for c in ['PAD790Q','PAD790U','PAD800']):
    df['MOD_ACTIVITY_MIN_WEEK'] = standardise_to_weekly('PAD790Q','PAD790U','PAD800', df)
    # Cap at 99th pct — extreme outliers likely reporting errors
    p99 = df['MOD_ACTIVITY_MIN_WEEK'].quantile(0.99)
    df['MOD_ACTIVITY_MIN_WEEK'] = df['MOD_ACTIVITY_MIN_WEEK'].clip(upper=p99)
    print(f'Moderate activity: mean={df["MOD_ACTIVITY_MIN_WEEK"].mean():.0f} min/week')

if all(c in df.columns for c in ['PAD810Q','PAD810U','PAD820']):
    df['VIG_ACTIVITY_MIN_WEEK'] = standardise_to_weekly('PAD810Q','PAD810U','PAD820', df)
    p99 = df['VIG_ACTIVITY_MIN_WEEK'].quantile(0.99)
    df['VIG_ACTIVITY_MIN_WEEK'] = df['VIG_ACTIVITY_MIN_WEEK'].clip(upper=p99)
    print(f'Vigorous activity: mean={df["VIG_ACTIVITY_MIN_WEEK"].mean():.0f} min/week')

# Drop raw unit columns after derivation
df.drop(columns=[c for c in ['PAD790Q','PAD790U','PAD800','PAD810Q','PAD810U','PAD820'] if c in df.columns], inplace=True)
print('Physical activity standardised to min/week.')

### 5g — Sleep Derived Features

Sleep debt (weekend catch-up sleep) is the difference between weekend and weekday sleep hours. Positive values indicate chronic weekday sleep restriction — the person sleeps longer on weekends to compensate. Chronic sleep debt is independently associated with metabolic dysregulation, insulin resistance, and higher BMI class.

In [ ]:
if 'SLD013' in df.columns and 'SLD012' in df.columns:
    df['SLEEP_DEBT'] = df['SLD013'] - df['SLD012']  # positive = catching up on weekends
    print('Sleep debt (weekend − weekday hours):')
    print(df['SLEEP_DEBT'].describe().round(2))

### 5h — Smoking Status (4-Level Derived Variable)

Adult smoking status in NHANES is not a single column. SMQ020 identifies whether someone ever smoked 100+ cigarettes (the standard ever-smoker threshold). SMQ040 asks current frequency among ever-smokers. Combining them gives a proper 4-level variable: never / former / current some-days / current daily.

We one-hot encode this with never-smoker as the reference category, producing three binary columns. Never-smoker is reference because it is the baseline health state.

In [ ]:
def derive_smoking_status(row):
    smq020 = row.get('SMQ020', np.nan)
    smq040 = row.get('SMQ040', np.nan)
    if pd.isna(smq020):
        return np.nan
    if smq020 == 2:
        return 'never_smoker'
    if smq020 == 1:
        if pd.isna(smq040):
            return np.nan
        if smq040 == 3:
            return 'former_smoker'
        if smq040 == 2:
            return 'current_somedays'
        if smq040 == 1:
            return 'current_daily'
    return np.nan

df['SMOKING_STATUS'] = df.apply(derive_smoking_status, axis=1)

print('Smoking status distribution:')
print(df['SMOKING_STATUS'].value_counts(dropna=False))

# One-hot encode — reference category: never_smoker (all three flags = 0)
smoking_dummies = pd.get_dummies(df['SMOKING_STATUS'], prefix='SMOKE')
# Keep only the three non-reference categories
for col in ['SMOKE_current_daily','SMOKE_current_somedays','SMOKE_former_smoker']:
    if col in smoking_dummies.columns:
        df[col] = smoking_dummies[col].astype(float)

# Drop raw smoking columns
df.drop(columns=[c for c in ['SMQ020','SMQ040','SMOKING_STATUS'] if c in df.columns], inplace=True)
print('\nSmoking one-hot columns created: SMOKE_current_daily, SMOKE_current_somedays, SMOKE_former_smoker')

### 5i — Weight History Derived Features

Three features derived from self-reported weight history:

- **Weight change over 1 year:** Positive = gained weight. OW_II participants are more likely to have gained or plateaued at higher weight compared to OW_I.
- **Self-reported BMI:** Computed from self-reported weight and height (WHD020 in lbs, WHD010 in inches). Used to compute the delta below.
- **BMI self-report delta:** Self-reported BMI minus measured BMI. OW_II individuals systematically under-report their weight more than OW_I individuals — this gap is itself a discriminating signal, reflecting both self-perception bias and social desirability effects.

In [ ]:
# Weight change over 1 year (pounds) — winsorise to reduce recall error noise
if 'WHD020' in df.columns and 'WHD050' in df.columns:
    df['WEIGHT_CHANGE_1YR'] = df['WHD020'] - df['WHD050']
    df['WEIGHT_CHANGE_1YR'] = df['WEIGHT_CHANGE_1YR'].clip(lower=-50, upper=50)
    print(f'Weight change 1yr — mean: {df["WEIGHT_CHANGE_1YR"].mean():.2f} lbs')

# Self-reported BMI = (weight_lbs / height_in^2) × 703
if 'WHD020' in df.columns and 'WHD010' in df.columns:
    # Filter implausible heights (< 48 inches or > 84 inches before computing)
    height_valid = df['WHD010'].between(48, 84)
    df['SELF_RPT_BMI'] = np.where(
        height_valid,
        (df['WHD020'] / (df['WHD010'] ** 2)) * 703,
        np.nan
    )

# BMI self-report delta = self-reported BMI − measured BMI
if 'SELF_RPT_BMI' in df.columns and 'BMXBMI' in df.columns:
    df['BMI_SELF_DELTA'] = df['SELF_RPT_BMI'] - df['BMXBMI']
    # Winsorise extreme deltas (likely data entry errors)
    df['BMI_SELF_DELTA'] = df['BMI_SELF_DELTA'].clip(lower=-10, upper=10)
    print(f'BMI self-report delta — mean: {df["BMI_SELF_DELTA"].mean():.2f} (negative = under-reports weight)')

# Incontinence Severity Index = KIQ005 (frequency) × KIQ010 (amount)
# Thresholds: 1-2=slight, 3-6=moderate, 8-9=severe, 12=very severe
if 'KIQ005' in df.columns and 'KIQ010' in df.columns:
    df['ISI_SCORE'] = df['KIQ005'].fillna(0) * df['KIQ010'].fillna(0)
    print(f'ISI score computed. Mean: {df["ISI_SCORE"].mean():.2f}')

### 5j — Alcohol Frequency Reordering

ALQ121 and ALQ142 use a 1–10 code where **1 = every day (highest frequency)** and **10 = 1-2 times per year (lowest frequency)**. The scale is NOT monotone — a higher code means less frequent drinking. This must be reordered before the model sees these features, otherwise the model learns the wrong direction entirely.

In [ ]:
# Recode so higher values = more frequent drinking
# Original: 0=never, 1=every day, 2=nearly every day, ..., 10=1-2 times/year
# Reordered: 0=never/0, 1=1-2 times/year, ..., 10=every day
freq_recode = {0: 0, 10: 1, 9: 2, 8: 3, 7: 4, 6: 5, 5: 6, 4: 7, 3: 8, 2: 9, 1: 10}

for col in ['ALQ121', 'ALQ142', 'ALQ270', 'ALQ280']:
    if col in df.columns:
        df[col] = df[col].map(freq_recode).where(df[col].notna(), df[col])
        df.rename(columns={col: f'{col}_freq'}, inplace=True)

print('ALQ frequency columns reordered to monotone scale (higher = more frequent).')

---
## 6. Encoding

### 6a — Binary Recodes (NHANES 1/2 → 0/1)

NHANES consistently codes Yes/No as 1/2. We recode these to 1/0. This must happen **after** refusal cleaning (so the 7/9 codes are already NaN) and **after** structural zero imputation.

In [ ]:
# Map: {1: 1, 2: 0} for standard Yes/No variables
binary_yes_no_cols = [
    'DMQMILIZ', 'DMDBORN4',
    'BPQ020', 'BPQ030', 'BPQ080', 'BPQ101D', 'BPQ150',
    'DIQ160', 'DIQ050', 'DIQ070',
    'MCQ010', 'MCQ160A', 'MCQ160B', 'MCQ160C', 'MCQ160D',
    'MCQ160E', 'MCQ160F', 'MCQ160L', 'MCQ160M', 'MCQ160P',
    'MCQ220', 'MCQ500', 'MCQ053', 'OSQ230',
    'KIQ022', 'KIQ042', 'KIQ044', 'KIQ052',
    'ALQ111', 'ALQ151',
    'WHQ070',
    'BAQ321A', 'BAQ321B', 'BAQ321C', 'BAQ321D',
    'BAQ391A', 'BAQ391B', 'BAQ401', 'BAQ431', 'BAQ491', 'BAQ560',
]

for col in binary_yes_no_cols:
    if col in df.columns:
        df[col] = df[col].map({1: 1, 2: 0})

# Special binary recodes with different mappings
if 'RIAGENDR' in df.columns:
    df['RIAGENDR'] = df['RIAGENDR'].map({1: 0, 2: 1})  # 0=male, 1=female

if 'RIDEXMON' in df.columns:
    df['RIDEXMON'] = df['RIDEXMON'].map({1: 0, 2: 1})  # 0=winter, 1=summer

print(f'Binary recoding complete: {len(binary_yes_no_cols)} columns recoded.')

### 6b — Ordinal Encoding

DIQ010 (diabetes status) is the most important ordinal variable. It has a genuine severity gradient: No < Borderline < Yes. We encode it as 0/1/2 respectively — one-hot encoding would lose the ordering information.

In [ ]:
# DIQ010: 2=No → 0, 3=Borderline → 1, 1=Yes → 2
if 'DIQ010' in df.columns:
    df['DIQ010'] = df['DIQ010'].map({2: 0, 3: 1, 1: 2})
    print('DIQ010 diabetes status: No=0, Borderline=1, Yes=2')
    print(df['DIQ010'].value_counts(dropna=False))

# KIQ005 urinary leakage frequency: Never=0, <monthly=1, few/month=2, few/week=3, daily=4
if 'KIQ005' in df.columns:
    df['KIQ005'] = df['KIQ005'].map({1: 0, 2: 1, 3: 2, 4: 3, 5: 4}).fillna(df['KIQ005'])

# BAQ341: how long had dizziness (1=less than 3 months ... 5=lifetime)
# Already 1-indexed and ordinal — leave as-is after NaN cleaning

# BAQ421: number of falls (ordinal bins 1-5: 1=1 fall, 5=5+ falls)
# Already ordinal — leave as-is

# BAQ530: unsteadiness frequency (1=never ... 6=all the time)
# Clean: 77/99 already replaced with NaN above

print('\nOrdinal encoding complete.')

### 6c — One-Hot Encoding

RIDRETH3 (race/ethnicity) and DMDMARTZ (marital status) are nominal — there is no meaningful ordering between categories. We one-hot encode them with a reference category dropped to avoid perfect multicollinearity.

In [ ]:
# RIDRETH3: reference category = 3 (Non-Hispanic White)
# Categories: 1=Mexican American, 2=Other Hispanic, 3=NH White, 4=NH Black, 6=NH Asian, 7=Other/Multi
if 'RIDRETH3' in df.columns:
    race_dummies = pd.get_dummies(df['RIDRETH3'], prefix='RACE')
    # Drop reference (NH White = code 3)
    ref_col = 'RACE_3.0' if 'RACE_3.0' in race_dummies.columns else 'RACE_3'
    race_dummies.drop(columns=[c for c in race_dummies.columns if '3' in c], inplace=True)
    df = pd.concat([df.drop(columns=['RIDRETH3']), race_dummies], axis=1)
    print(f'RIDRETH3 one-hot columns: {[c for c in race_dummies.columns]}')

# DMDMARTZ: reference category = 3 (Never married)
# Categories: 1=Married/Partner, 2=Widowed/Divorced/Separated, 3=Never married
if 'DMDMARTZ' in df.columns:
    marital_dummies = pd.get_dummies(df['DMDMARTZ'], prefix='MARITAL')
    marital_dummies.drop(columns=[c for c in marital_dummies.columns if '3' in c], inplace=True)
    df = pd.concat([df.drop(columns=['DMDMARTZ']), marital_dummies], axis=1)
    print(f'DMDMARTZ one-hot columns: {[c for c in marital_dummies.columns]}')

print('\nOne-hot encoding complete.')

---
## 7. Drop Columns

Three categories of columns to drop:

1. **Data leakage:** BMXWT and BMXHT directly enter the BMI formula. Including them as features would let the model reconstruct the target variable from inputs — a form of label leakage that produces artificially high performance on training data but fails in production.
2. **SI-unit duplicates:** Every lab variable in NHANES comes in two units (e.g. mg/dL and mmol/L). These are exact linear transformations of each other — keeping both doubles the column count with zero additional information.
3. **Zero-variance columns:** Several columns had only one unique value across the entire dataset. These carry no discriminative information.

In [ ]:
cols_to_drop = [
    # Data leakage — enter BMI formula directly
    'BMXWT', 'BMXHT',
    # Pediatric-only exam variables
    'BMXRECUM', 'BMXHEAD',
    # QC flags (zero-variance or near-zero)
    'SDDSRVYR', 'BMIRECUM', 'BMILEG', 'BMIARML', 'BMIARMC', 'BMIWAIST', 'BMIHIP',
    'BMDSTATS', 'BMIWT', 'BMIHT', 'BMIHEAD',
    # SI unit duplicates — keep mg/dL or primary US unit versions
    'LBDSALSI', 'LBDSBUSI', 'LBDSCRSI', 'LBDSGBSI', 'LBDSGLSI',
    'LBDSCHSI', 'LBDSTRSI', 'LBDSUASI', 'LBDSTPSI', 'LBDSTBSI',
    'LBDSPHSI', 'LBDSIRSI', 'LBDSCASI', 'LBDINSI', 'LBDGLUSI', 'LBDTCSI',
    # Comment/flag columns (already handled)
    'LBDSGTLC', 'LBDSTBLC', 'LBDINLC',
    # Osmolality (derived from other kept variables)
    'LBXSOSSI',
    # Survey design variables (not predictors)
    'SDMVSTRA', 'SDMVPSU', 'WTINT2YR', 'WTMEC2YR', 'WTSAF2YR', 'WTPH2YR',
    # Identifiers
    'SEQN', 'RIDSTATR',
    # Superseded variables
    'RIDRETH1', 'RIDAGEMN', 'RIDEXAGM', 'RIDEXPRG',
    # Household reference person (only for under-20)
    'DMDHRGND', 'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ',
    # HSQ_L — zero relevance to OW classification
    'HSQ590',
    # MCQ redundant/sparse
    'MCQ035','MCQ040','MCQ050','MCQ149','MCQ170M','MCQ170L',
    'MCQ510B','MCQ510C','MCQ510D','MCQ510E','MCQ510F','MCQ230D',
    'MCQ510A', 'MCQ550', 'MCQ560',
    # DIQ redundant
    'DID040', 'DID060', 'DIQ060U', 'DIQ180',
    # SMQ youth-only / zero-variance after adult filter
    'SMD100MN', 'SMQ621', 'SMD630', 'SMAQUEX2',
    # KIQ near-zero variance
    'KIQ025',
    # Self-reported weight/height raw cols (derived features already computed)
    'WHD010', 'WHD020', 'WHD050', 'SELF_RPT_BMI',
    # BIOPRO glucose/cholesterol/triglycerides — use reference files instead
    'LBXSGL', 'LBXSCH', 'LBXSTR',
]

before = df.shape[1]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
after = df.shape[1]
print(f'Dropped {before - after} columns. Remaining: {after}')

---
## 8. Target Variable & Final Dataset Shape

We define OW_CLASS using WHO/CDC adult BMI thresholds:
- **OW_I** — BMI 25.0 to 29.9 (Overweight)
- **OW_II** — BMI 30.0 and above (Obese)

Participants with BMI below 25 are excluded — they will never trigger the Model 2 router.

In [ ]:
df['OW_CLASS'] = pd.cut(
    df['BMXBMI'],
    bins=[25, 30, np.inf],
    labels=['OW_I', 'OW_II'],
    right=False
)

# Working dataset: only OW participants
ow = df[df['OW_CLASS'].notna()].copy()

print('Full filtered dataset (all BMI):')
print(f'  Rows: {len(df):,}  |  Columns: {df.shape[1]}')
print(f'\nOW working set (BMI >= 25):')
print(f'  Rows: {len(ow):,}')
print(ow['OW_CLASS'].value_counts())
print(ow['OW_CLASS'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

usable_fasting = ow[ow['LBXGLU'].notna()]
print(f'\nUsable for full clinical model (OW + fasting glucose): {len(usable_fasting):,}')
print(usable_fasting['OW_CLASS'].value_counts())

---
## 9. Gender Split Feasibility Analysis

Body composition, fat distribution, and metabolic markers differ substantially between males and females. A model trained on the pooled sample must learn these differences implicitly via the sex flag — but a sex-stratified model can learn the OW_I/OW_II boundary independently within each group, potentially with higher precision.

The key question is whether the subgroup sample sizes are large enough to train reliable separate models. For XGBoost with ~20–40 features, a working rule of thumb is a minimum of ~500–800 training rows with good class balance. We evaluate this below.

We also check whether the class balance, feature distributions, and comorbidity profiles differ enough between males and females to justify the additional complexity.

In [ ]:
# RIAGENDR was recoded: 0=male, 1=female
male_ow   = ow[ow['RIAGENDR'] == 0].copy()
female_ow = ow[ow['RIAGENDR'] == 1].copy()

print('='*55)
print('GENDER SPLIT — Sample Sizes')
print('='*55)
for label, subset in [('Male', male_ow), ('Female', female_ow)]:
    n_total = len(subset)
    n_fasting = subset['LBXGLU'].notna().sum()
    class_counts = subset['OW_CLASS'].value_counts()
    class_pct    = subset['OW_CLASS'].value_counts(normalize=True).mul(100).round(1)
    imbalance    = class_pct.max() / class_pct.min()

    print(f'\n{label}:')
    print(f'  Total OW participants:     {n_total:,}')
    print(f'  With fasting glucose:      {n_fasting:,}  ({n_fasting/n_total*100:.0f}% of OW)')
    print(f'  OW_I:  {class_counts.get("OW_I",0):,}  ({class_pct.get("OW_I",0)}%)')
    print(f'  OW_II: {class_counts.get("OW_II",0):,}  ({class_pct.get("OW_II",0)}%)')
    print(f'  Imbalance ratio:           {imbalance:.2f}x')
    if n_fasting >= 800:
        print(f'  Verdict: ✅ SUFFICIENT for separate model (n_fasting >= 800)')
    elif n_fasting >= 500:
        print(f'  Verdict: ⚠️  BORDERLINE — viable but tighter confidence intervals')
    else:
        print(f'  Verdict: ❌ TOO SMALL — pool with full dataset')

In [ ]:
from scipy import stats

# Test whether key features show different OW_I vs OW_II separation in males vs females
# If the effect sizes are substantially different, sex-specific models are more justified

key_features = ['BMXWAIST', 'SYS_BP', 'DIA_BP', 'LBXGH', 'HOMA_IR_log',
                'WAIST_HT_RATIO', 'WAIST_HIP_RATIO', 'eGFR',
                'LBXGLU', 'ALT_log', 'LBXSUA']

rows = []
for feat in key_features:
    for gender_label, subset in [('Male', male_ow), ('Female', female_ow)]:
        if feat not in subset.columns:
            continue
        g1 = subset[subset['OW_CLASS']=='OW_I'][feat].dropna()
        g2 = subset[subset['OW_CLASS']=='OW_II'][feat].dropna()
        if len(g1) < 30 or len(g2) < 30:
            continue
        stat, pval = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        effect = abs(1 - (2*stat)/(len(g1)*len(g2)))
        rows.append({
            'Feature': feat,
            'Gender': gender_label,
            'OW_I mean': round(g1.mean(), 2),
            'OW_II mean': round(g2.mean(), 2),
            'Effect size (r)': round(effect, 3),
            'p-value': round(pval, 4)
        })

sep_df = pd.DataFrame(rows)

# Pivot to show male vs female effect side by side
pivot = sep_df.pivot_table(
    index='Feature', columns='Gender', values='Effect size (r)'
).round(3)
pivot['Difference'] = (pivot['Male'] - pivot['Female']).abs().round(3)
pivot = pivot.sort_values('Difference', ascending=False)

print('Feature separability by gender (Mann-Whitney effect size):')
print('Large difference = feature behaves differently in males vs females')
print(pivot.to_string())

In [ ]:
# Visualise the three most divergent features
top_divergent = pivot.head(3).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Feature Distributions by OW Class — Male vs Female', fontsize=13)

for row_idx, gender_label, subset in [(0, 'Male', male_ow), (1, 'Female', female_ow)]:
    for col_idx, feat in enumerate(top_divergent):
        ax = axes[row_idx][col_idx]
        sub = subset[[feat, 'OW_CLASS']].dropna()
        for cls, color in [('OW_I','#4C72B0'),('OW_II','#DD8452')]:
            vals = sub[sub['OW_CLASS']==cls][feat]
            ax.hist(vals, bins=30, alpha=0.55, density=True, color=color, label=cls)
        ax.set_title(f'{gender_label} — {feat}', fontsize=9)
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('merged_data/gender_feature_divergence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comorbidity prevalence by class — compare male vs female gap
comorbidity_cols = [
    ('DIQ010',   'Diabetes (ordinal)'),
    ('BPQ020',   'Hypertension dx'),
    ('BPQ101D',  'Cholesterol meds'),
    ('MCQ160C',  'Coronary HD'),
    ('MCQ160E',  'Heart attack'),
    ('MCQ160A',  'Arthritis'),
    ('MCQ160L',  'Liver condition'),
    ('KIQ042',   'Stress incontinence'),
]

print('Comorbidity prevalence — OW_II vs OW_I gap by gender:')
print(f'{"Condition":25} {"Male gap":12} {"Female gap":12} {"Divergence"}')
print('-'*60)

for col, label in comorbidity_cols:
    if col not in ow.columns:
        continue
    gaps = {}
    for gender_label, subset in [('Male', male_ow), ('Female', female_ow)]:
        sub = subset[[col, 'OW_CLASS']].dropna()
        sub = sub[sub[col].isin([0,1,2])]
        if sub.empty:
            gaps[gender_label] = np.nan
            continue
        # For ordinal DIQ010 use mean; for binary use proportion
        if col == 'DIQ010':
            g1 = sub[sub['OW_CLASS']=='OW_I'][col].mean()
            g2 = sub[sub['OW_CLASS']=='OW_II'][col].mean()
        else:
            g1 = (sub[sub['OW_CLASS']=='OW_I'][col]==1).mean() * 100
            g2 = (sub[sub['OW_CLASS']=='OW_II'][col]==1).mean() * 100
        gaps[gender_label] = round(g2 - g1, 2)

    m_gap = gaps.get('Male', np.nan)
    f_gap = gaps.get('Female', np.nan)
    div = '' if pd.isna(m_gap) or pd.isna(f_gap) else f'{abs(m_gap - f_gap):.2f}'
    flag = ' ⚠️ ' if not pd.isna(m_gap) and not pd.isna(f_gap) and abs(m_gap - f_gap) > 8 else ''
    print(f'{label:25} {str(m_gap):12} {str(f_gap):12} {div}{flag}')

In [ ]:
print('='*60)
print('GENDER SPLIT VERDICT')
print('='*60)

n_male_fasting   = male_ow['LBXGLU'].notna().sum()
n_female_fasting = female_ow['LBXGLU'].notna().sum()

print(f"""
Male fasting N:   {n_male_fasting:,}
Female fasting N: {n_female_fasting:,}

Recommendation:
""")

if n_male_fasting >= 800 and n_female_fasting >= 800:
    print("""✅ TRAIN SEPARATE MODELS

Both subgroups exceed 800 fasting participants — sufficient for
reliable XGBoost training with ~30-50 features.

Justification:
- Body fat distribution mechanisms differ by sex (gynoid vs android)
- Waist circumference thresholds for metabolic risk differ by sex
- Incontinence features are predominantly female-specific signal
- Uric acid and creatinine have sex-specific reference ranges
- Feature effect sizes show measurable divergence across genders

Deployment approach:
  Router checks RIAGENDR and routes to male_model or female_model
  Keep the pooled model as a fallback for missing sex data
""")
elif n_male_fasting >= 500 and n_female_fasting >= 500:
    print("""⚠️  BORDERLINE — TRAIN BOTH BUT VALIDATE CAREFULLY

Sample sizes are workable but confidence intervals will be wider.
Use conservative regularisation (lower learning rate, higher min_child_weight).
Run cross-validation with at least 5 folds stratified on OW_CLASS.
""")
else:
    print("""❌ POOL — insufficient N for reliable sex-stratified models.
Keep RIAGENDR as a feature in the pooled model instead.
""")

---
## 10. Feature Summary & Export

Final summary of the engineered dataset. We export four files:
- `ow_full.csv` — all OW participants with all engineered features (some missing for non-fasting)
- `ow_fasting.csv` — OW participants with fasting glucose (the primary Model 2 training set)
- `ow_male_fasting.csv` — male-only fasting subset
- `ow_female_fasting.csv` — female-only fasting subset

In [ ]:
# Final feature list — all columns except target
feature_cols = [c for c in ow.columns if c not in ['OW_CLASS', 'BMXBMI']]

print(f'Total features after engineering: {len(feature_cols)}')
print(f'Target: OW_CLASS (OW_I=0, OW_II=1)')

# Missingness summary on final feature set
miss = ow[feature_cols].isna().mean().mul(100).round(1)
miss_df = miss.reset_index()
miss_df.columns = ['feature', 'null_pct']
miss_df = miss_df.sort_values('null_pct', ascending=False)

print('\nTop 20 most missing features:')
print(miss_df.head(20).to_string(index=False))

print(f'\nFeatures with 0% missing: {(miss==0).sum()}')
print(f'Features with <20% missing: {(miss<20).sum()}')
print(f'Features with 20-50% missing: {((miss>=20) & (miss<50)).sum()}')
print(f'Features with >50% missing: {(miss>=50).sum()}')

In [ ]:
os.makedirs('merged_data', exist_ok=True)

# Full OW set
ow.to_csv('merged_data/ow_full.csv', index=False)
print(f'Saved ow_full.csv — {len(ow):,} rows × {ow.shape[1]} cols')

# Fasting subset (primary training set)
ow_fasting = ow[ow['LBXGLU'].notna()].copy()
ow_fasting.to_csv('merged_data/ow_fasting.csv', index=False)
print(f'Saved ow_fasting.csv — {len(ow_fasting):,} rows × {ow_fasting.shape[1]} cols')

# Sex-stratified fasting subsets
ow_male   = ow_fasting[ow_fasting['RIAGENDR'] == 0].copy()
ow_female = ow_fasting[ow_fasting['RIAGENDR'] == 1].copy()

ow_male.to_csv('merged_data/ow_male_fasting.csv', index=False)
ow_female.to_csv('merged_data/ow_female_fasting.csv', index=False)
print(f'Saved ow_male_fasting.csv   — {len(ow_male):,} rows')
print(f'Saved ow_female_fasting.csv — {len(ow_female):,} rows')

# Save feature list
feature_meta = {
    'feature_cols': feature_cols,
    'target': 'OW_CLASS',
    'n_features': len(feature_cols),
    'n_ow_full': len(ow),
    'n_ow_fasting': len(ow_fasting),
    'n_male_fasting': len(ow_male),
    'n_female_fasting': len(ow_female),
}
with open('merged_data/feature_meta.json', 'w') as f:
    json.dump(feature_meta, f, indent=2)

print('\nAll exports complete. Ready for model training.')

---
## Summary

**What this notebook produced:**

Starting from the raw merged NHANES data, we applied a full preprocessing pipeline:
- Mandatory filters: MEC-examined adults, age 20+, non-pregnant
- NHANES refusal/DK code cleaning across all questionnaire variables
- Structural zero imputation for skip-pattern variables (alcohol, BP meds, incontinence, diabetes meds, smoking)
- Feature engineering: body composition ratios, BP averaging, HOMA-IR, eGFR, biochemistry ratios, physical activity standardisation, sleep debt, smoking status derivation, weight history features, ALQ frequency reordering
- Binary, ordinal, and one-hot encoding
- Column dropping: data leakage, SI-unit duplicates, zero-variance, administrative variables

**Gender split decision:** See Section 9 output. If both subgroups have ≥800 fasting participants, train separate male and female models — the biological justification is strong (fat distribution mechanisms, sex-specific reference ranges, incontinence features). If borderline, train both but use conservative regularisation and validate carefully.

**Next steps (model training notebook):**
1. Load `ow_fasting.csv` (or sex-stratified versions)
2. Train/test split stratified on OW_CLASS (80/20)
3. Handle remaining missingness (iterative imputation or XGBoost native NaN handling)
4. Train XGBoost binary classifier with `scale_pos_weight` for mild class imbalance
5. Evaluate AUC-ROC, precision-recall curve, confusion matrix
6. Run SHAP feature importance to confirm key discriminators
7. Tune the router threshold using the calibrated probability outputs

# Removing nan values

In [ ]:
df = pd.read_csv('merged_data/ow_fasting.csv')

MISS_THRESHOLD = 0.40

# ── 1. Columns above threshold ────────────────────────────────────────────────
miss_pct  = df.isnull().mean().mul(100).round(1)
high_miss = miss_pct[miss_pct > MISS_THRESHOLD * 100].sort_values(ascending=False)

print(f"Columns with >{MISS_THRESHOLD*100:.0f}% missingness: {len(high_miss)}\n")
print(f"{'Column':<30} {'Missing%':>9}  {'Non-null N':>10}")
print("-" * 55)
for col, pct in high_miss.items():
    n = df[col].notna().sum()
    print(f"{col:<30} {pct:>8.1f}%  {n:>10,}")

# ── 2. Point-biserial correlation with target for each high-miss column ────────
# OW_CLASS: OW_I=0, OW_II=1
target = (df['OW_CLASS'] == 'OW_II').astype(float)

print(f"\n{'Column':<30} {'Corr w/ target':>15}  {'Non-null N':>10}  {'Missing%':>9}")
print("-" * 70)

results = []
for col in high_miss.index:
    if col in ['OW_CLASS', 'BMXBMI']:
        continue
    if df[col].dtype == object:
        continue
    valid = df[col].notna()
    if valid.sum() < 30:
        corr = np.nan
    else:
        corr = df.loc[valid, col].corr(target[valid])
    results.append({'col': col, 'corr': corr,
                    'n': valid.sum(), 'miss_pct': high_miss[col]})

results = sorted(results, key=lambda x: abs(x['corr']) if not np.isnan(x['corr']) else -1, reverse=True)

for r in results:
    corr_str = f"{r['corr']:+.3f}" if not np.isnan(r['corr']) else "   N/A"
    print(f"{r['col']:<30} {corr_str:>15}  {r['n']:>10,}  {r['miss_pct']:>8.1f}%")

dropping those and creating male female

In [ ]:
MISS_THRESHOLD = 0.40

for fname in ['ow_fasting', 'ow_male_fasting', 'ow_female_fasting']:
    df = pd.read_csv(f'merged_data/{fname}.csv')

    miss_pct  = df.isnull().mean()
    drop_cols = miss_pct[miss_pct > MISS_THRESHOLD].index.tolist()
    drop_cols = [c for c in drop_cols if c not in ['OW_CLASS', 'BMXBMI']]

    df.drop(columns=drop_cols, inplace=True)
    df.to_csv(f'merged_data/{fname}_clean.csv', index=False)

    print(f"{fname}_clean.csv")
    print(f"  Dropped:   {len(drop_cols)} columns")
    print(f"  Remaining: {df.shape[1]} columns, {df.shape[0]:,} rows")
    print(f"  OW_CLASS:  {df['OW_CLASS'].value_counts().to_dict()}")
    print()